# Data Sprint 2.0 (Anveshan, NIE Mysuru) — NLP Track
## Script-Bridged, Shift-Robust Hinglish Intent Classifier

- **Primary Metric:** Unweighted Macro-F1 over 56 classes (55 in-scope domains + Class 55 Fallback)
- **Hard Constraints:** <= 300M total parameters | < 50ms inference latency per query (including transliteration and TTA)
- **Validation Framework:** Stratified Group 5-Fold cross-validation on connected-component twin clusters (disjoint across splits)
- **Pipeline:** Dual-View Phonetic Transliteration Bridge + Character & Word N-gram Linear Classifiers + Prior Logit Adjustment + Two-Tier Fallback


In [1]:
# 1. SETUP & CONFIGURATION
import os
import sys
import time
import math
import re
import unicodedata
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from scipy.sparse import hstack
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

SEED = 42
np.random.seed(SEED)
print(f"Random seed fixed to {SEED}. Environment initialized.")


Random seed fixed to 42. Environment initialized.


## 2. Data Loading & Script Shift Diagnostics
Verify shapes, ranges, duplicates, and check the massive script distribution shift between Train and Test.


In [2]:
train_path = 'train.csv' if os.path.exists('train.csv') else '/kaggle/input/anveshan-data-sprint-2-nlp/train.csv'
test_path = 'test.csv' if os.path.exists('test.csv') else '/kaggle/input/anveshan-data-sprint-2-nlp/test.csv'
sample_path = 'sample_submission.csv' if os.path.exists('sample_submission.csv') else '/kaggle/input/anveshan-data-sprint-2-nlp/sample_submission.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

def classify_script(text):
    has_dev = any('\u0900' <= ch <= '\u097F' for ch in text)
    has_lat = any('a' <= ch.lower() <= 'z' for ch in text)
    if has_dev and has_lat: return 'mixed'
    elif has_dev: return 'devanagari'
    elif has_lat: return 'roman'
    return 'other'

train_df['script'] = train_df['query_text'].apply(classify_script)
test_df['script'] = test_df['query_text'].apply(classify_script)

print(f"Train Shape: {train_df.shape}, Test Shape: {test_df.shape}")
print("\nTrain Script Breakdown:")
print(train_df['script'].value_counts(normalize=True))
print("\nTest Script Breakdown:")
print(test_df['script'].value_counts(normalize=True))


Train Shape: (16936, 4), Test Shape: (4912, 3)

Train Script Breakdown:
script
devanagari    0.611301
roman         0.333314
mixed         0.055385
Name: proportion, dtype: float64

Test Script Breakdown:
script
roman         0.642101
devanagari    0.326547
mixed         0.031352
Name: proportion, dtype: float64


## 3. Transliteration Bridge (N1)
Calibrated phonetic Devanagari -> Roman mapper that mirrors the romanization conventions observed in the test set.


In [3]:
VOWELS = {
    'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ee', 'उ': 'u', 'ऊ': 'oo',
    'ऋ': 'ri', 'ए': 'e', 'ऐ': 'ai', 'ओ': 'o', 'औ': 'au',
    'ऑ': 'o', 'ऍ': 'e'
}
MATRAS = {
    'ा': 'aa', 'ि': 'i', 'ी': 'ee', 'ु': 'u', 'ू': 'oo',
    'ृ': 'ri', 'े': 'e', 'ै': 'ai', 'ो': 'o', 'ौ': 'au',
    'ॉ': 'o', 'ॅ': 'e'
}
CONSONANTS = {
    'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ng',
    'च': 'ch', 'छ': 'chh', 'ज': 'j', 'झ': 'jh', 'ञ': 'ny',
    'ट': 't', 'ठ': 'th', 'ड': 'd', 'ढ': 'dh', 'ण': 'n',
    'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
    'प': 'p', 'फ': 'ph', 'ब': 'b', 'भ': 'bh', 'म': 'm',
    'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'v',
    'श': 'sh', 'ष': 'sh', 'स': 's', 'ह': 'h',
    'क़': 'q', 'ख़': 'kh', 'ग़': 'gh', 'ज़': 'z', 'ड़': 'd', 'ढ़': 'dh', 'फ़': 'f'
}
HALANT = '्'
ANUSVARA = 'ं'
CHANDRABINDU = 'ँ'
VISARGA = 'ः'
NUKTA = '़'

def devanagari_to_roman(text):
    text = str(text)
    res = []
    i = 0
    n = len(text)
    while i < n:
        c = text[i]
        if i + 1 < n and text[i+1] == NUKTA:
            c = c + NUKTA
            i += 1
        if c in VOWELS:
            res.append(VOWELS[c])
        elif c in CONSONANTS:
            base = CONSONANTS[c]
            if i + 1 < n:
                nxt = text[i+1]
                if nxt == HALANT:
                    res.append(base); i += 1
                elif nxt in MATRAS:
                    res.append(base + MATRAS[nxt]); i += 1
                elif nxt in [ANUSVARA, CHANDRABINDU]:
                    res.append(base + 'a')
                elif nxt in CONSONANTS or nxt in VOWELS:
                    res.append(base + 'a')
                elif nxt.isspace() or nxt in '.,!?"\'()[]-;:':
                    res.append(base)
                else:
                    res.append(base + 'a')
            else:
                res.append(base)
        elif c == ANUSVARA or c == CHANDRABINDU:
            res.append('n')
        elif c == VISARGA:
            res.append('h')
        elif c in MATRAS:
            res.append(MATRAS[c])
        else:
            res.append(c)
        i += 1
    return ''.join(res)

train_df['trans'] = train_df['query_text'].apply(
    lambda x: devanagari_to_roman(x) if any('\u0900' <= ch <= '\u097F' for ch in x) else x
)
test_df['trans'] = test_df['query_text'].apply(
    lambda x: devanagari_to_roman(x) if any('\u0900' <= ch <= '\u097F' for ch in x) else x
)

# Demonstration of phonetic transliteration
sample_devs = [
    'कृपया रसोईघर की बत्तियां बंद कर दें',
    'आधा करने के लिए कम मात्रा',
    'शतरंज खेल शुरू करो'
]
for s in sample_devs:
    print(f"{s}  ==>  {devanagari_to_roman(s)}")


कृपया रसोईघर की बत्तियां बंद कर दें  ==>  kripayaa rasoeeghar kee battiyaan band kar den
आधा करने के लिए कम मात्रा  ==>  aadhaa karane ke lie kam maatraa
शतरंज खेल शुरू करो  ==>  shataranj khel shuroo karo


## 4. Non-Leaky Twin-Grouped Cross-Validation (N3)
Cross-script twin pairs (parallel translations) share identical intents across scripts. Standard random or stratified K-Fold allows twins on both sides of a fold, causing optimistic leakage. We build connected-component clusters of near-twins and use `StratifiedGroupKFold` so entire clusters remain strictly disjoint across splits.


In [4]:
vec_grp = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), min_df=1)
X_grp = vec_grp.fit_transform(train_df['trans'])
nn = NearestNeighbors(n_neighbors=15, metric='cosine')
nn.fit(X_grp)
dists, indices = nn.kneighbors(X_grp)

rows, cols = [], []
for i in range(len(train_df)):
    for j, d in zip(indices[i][1:], dists[i][1:]):
        if d < 0.25: # cosine similarity >= 0.75
            rows.append(i); cols.append(j)

adj = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(train_df), len(train_df)))
_, labels = connected_components(adj, directed=False)
train_df['group_id'] = labels
print(f"Total twin clusters formed: {len(set(labels))} out of {len(train_df)} rows.")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_df['fold'] = -1
for fold, (trn_idx, val_idx) in enumerate(sgkf.split(train_df, train_df['target'], train_df['group_id'])):
    train_df.loc[val_idx, 'fold'] = fold


Total twin clusters formed: 11946 out of 16936 rows.


## 5. Dual-View 5-Fold Training & Honest OOF Evaluation (N2)
Train on dual views (original + transliterated Devanagari) and perform Test-Time Augmentation (TTA) averaging at inference.


In [5]:
cache_file = 'twin_grouped_cache.npz'
if os.path.exists(cache_file):
    print("Loading cached 5-fold predictions...")
    cache = np.load(cache_file)
    oof_probs = cache['oof']
    test_probs = cache['test']
else:
    oof_probs = np.zeros((len(train_df), 56))
    test_probs = np.zeros((len(test_df), 56))

    for fold in range(5):
        trn = train_df[train_df['fold'] != fold]
        val = train_df[train_df['fold'] == fold]
        val_idx = val.index
        
        # Dual view augmentation on training set
        aug_rows = trn[trn['script'].isin(['devanagari', 'mixed'])].copy()
        aug_rows['query_text'] = aug_rows['trans']
        trn_aug = pd.concat([trn, aug_rows], ignore_index=True)
        
        vec_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)
        vec_w = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        
        X_tr = hstack([vec_c.fit_transform(trn_aug['query_text']), vec_w.fit_transform(trn_aug['query_text'])])
        clf = LogisticRegression(C=6.0, max_iter=1000, random_state=SEED)
        clf.fit(X_tr, trn_aug['target'])
        
        # Dual-view validation (TTA)
        X_va_o = hstack([vec_c.transform(val['query_text']), vec_w.transform(val['query_text'])])
        X_va_t = hstack([vec_c.transform(val['trans']), vec_w.transform(val['trans'])])
        oof_probs[val_idx] = 0.5 * clf.predict_proba(X_va_o) + 0.5 * clf.predict_proba(X_va_t)
        
        # Dual-view test
        X_te_o = hstack([vec_c.transform(test_df['query_text']), vec_w.transform(test_df['query_text'])])
        X_te_t = hstack([vec_c.transform(test_df['trans']), vec_w.transform(test_df['trans'])])
        test_probs += (0.5 * clf.predict_proba(X_te_o) + 0.5 * clf.predict_proba(X_te_t)) / 5.0

# Honest non-leaky OOF evaluation
preds_raw = np.argmax(oof_probs, axis=1)
rom_m = train_df['script'] == 'roman'
dev_m = train_df['script'] == 'devanagari'
mix_m = train_df['script'] == 'mixed'

f1_overall_raw = f1_score(train_df['target'], preds_raw, average='macro')
f1_rom_raw = f1_score(train_df.loc[rom_m, 'target'], preds_raw[rom_m.values], average='macro')
f1_dev_raw = f1_score(train_df.loc[dev_m, 'target'], preds_raw[dev_m.values], average='macro')
f1_mix_raw = f1_score(train_df.loc[mix_m, 'target'], preds_raw[mix_m.values], average='macro')
f1_test_weighted_raw = 0.642 * f1_rom_raw + 0.327 * f1_dev_raw + 0.031 * f1_mix_raw

print("--- NON-LEAKY TWIN-GROUPED OOF METRICS (RAW) ---")
print(f"Overall Macro-F1: {f1_overall_raw:.4f}")
print(f"Roman-only Macro-F1: {f1_rom_raw:.4f}")
print(f"Devanagari-only Macro-F1: {f1_dev_raw:.4f}")
print(f"Mixed Macro-F1: {f1_mix_raw:.4f}")
print(f"Test-Composition Weighted Macro-F1: {f1_test_weighted_raw:.4f}")


Loading cached 5-fold predictions...
--- NON-LEAKY TWIN-GROUPED OOF METRICS (RAW) ---
Overall Macro-F1: 0.8082
Roman-only Macro-F1: 0.8139
Devanagari-only Macro-F1: 0.8037
Mixed Macro-F1: 0.7395
Test-Composition Weighted Macro-F1: 0.8083


## 6. Prior Logit Adjustment (N5) & Fallback Calibration (N4)
Handle severe class imbalance (from 1,224 rows to 84 rows) using logit adjustment (Menon et al., 2021).


In [6]:
class_counts = train_df['target'].value_counts().sort_index().values
prior = class_counts / len(train_df)
tau = 0.200

adj_oof_probs = oof_probs * (1.0 / (prior ** tau))
preds_adj = np.argmax(adj_oof_probs, axis=1)

f1_overall_adj = f1_score(train_df['target'], preds_adj, average='macro')
f1_rom_adj = f1_score(train_df.loc[rom_m, 'target'], preds_adj[rom_m.values], average='macro')
f1_dev_adj = f1_score(train_df.loc[dev_m, 'target'], preds_adj[dev_m.values], average='macro')
f1_mix_adj = f1_score(train_df.loc[mix_m, 'target'], preds_adj[mix_m.values], average='macro')
f1_test_weighted_adj = 0.642 * f1_rom_adj + 0.327 * f1_dev_adj + 0.031 * f1_mix_adj

print("--- ADJUSTED OOF METRICS (tau = 0.200) ---")
print(f"Overall Macro-F1: {f1_overall_adj:.4f} (+{f1_overall_adj - f1_overall_raw:.4f})")
print(f"Roman-only Macro-F1: {f1_rom_adj:.4f}")
print(f"Devanagari-only Macro-F1: {f1_dev_adj:.4f}")
print(f"Mixed Macro-F1: {f1_mix_adj:.4f}")
print(f"Test-Composition Weighted Macro-F1: {f1_test_weighted_adj:.4f}")

# Fallback Evaluation
y_true_55 = (train_df['target'] == 55).values
y_pred_55 = (preds_adj == 55)
print(f"Tier 1 Fallback (Class 55) OOF Precision: {precision_score(y_true_55, y_pred_55):.4f}")
print(f"Tier 1 Fallback (Class 55) OOF Recall:    {recall_score(y_true_55, y_pred_55):.4f}")
print(f"Tier 1 Fallback (Class 55) OOF F1-Score:  {f1_score(y_true_55, y_pred_55):.4f}")


--- ADJUSTED OOF METRICS (tau = 0.200) ---
Overall Macro-F1: 0.8157 (+0.0075)
Roman-only Macro-F1: 0.8257
Devanagari-only Macro-F1: 0.8097
Mixed Macro-F1: 0.7276
Test-Composition Weighted Macro-F1: 0.8174
Tier 1 Fallback (Class 55) OOF Precision: 0.8652
Tier 1 Fallback (Class 55) OOF Recall:    0.7992
Tier 1 Fallback (Class 55) OOF F1-Score:  0.8309


## 7. Full-Dataset Production Model & Real Inference `predict()` Pipeline
Train on the entire training dataset with dual augmentation to build the deployable pipeline.


In [7]:
aug_rows = train_df[train_df['script'].isin(['devanagari', 'mixed'])].copy()
aug_rows['query_text'] = aug_rows['trans']
train_full = pd.concat([train_df, aug_rows], ignore_index=True)

vec_c_prod = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)
vec_w_prod = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True)

X_full = hstack([vec_c_prod.fit_transform(train_full['query_text']), vec_w_prod.fit_transform(train_full['query_text'])])
clf_prod = LogisticRegression(C=6.0, max_iter=1000, random_state=SEED)
clf_prod.fit(X_full, train_full['target'])

weights_prod = 1.0 / (prior ** tau)

def predict(text: str) -> dict:
    t0 = time.perf_counter()
    has_dev = any('\u0900' <= ch <= '\u097F' for ch in text)
    trans = devanagari_to_roman(text) if has_dev else text
    
    if len(text) > 250:
        return {
            'label': 55,
            'confidence': 1.0,
            'latency_ms': (time.perf_counter() - t0) * 1000,
            'is_fallback': True,
            'reason': 'open_set_long_prose'
        }
        
    feat_orig = hstack([vec_c_prod.transform([text]), vec_w_prod.transform([text])])
    feat_trans = hstack([vec_c_prod.transform([trans]), vec_w_prod.transform([trans])])
    
    p_orig = clf_prod.predict_proba(feat_orig)[0]
    p_trans = clf_prod.predict_proba(feat_trans)[0]
    p_dual = 0.5 * p_orig + 0.5 * p_trans
    
    p_adj = p_dual * weights_prod
    p_norm = p_adj / p_adj.sum()
    pred_label = int(np.argmax(p_norm))
    conf = float(np.max(p_norm))
    latency_ms = (time.perf_counter() - t0) * 1000
    
    return {
        'label': pred_label,
        'confidence': conf,
        'latency_ms': latency_ms,
        'is_fallback': (pred_label == 55)
    }

# Interactive Demonstration
test_queries = [
    "kripaya rasoeeghar ki battiyaan band kar den",
    "कृपया रसोईघर की बत्तियां बंद कर दें",
    "kya kal subah ke liye koi alarm set hai mera bhai",
    "bina tel ke svadisht snaiks kaise pakaaen",
    "volyum star ko adhiktam mein badlen"
]
print("--- LIVE INFERENCE DEMO ---")
for q in test_queries:
    res = predict(q)
    print(f"Query: {q}")
    print(f"  -> Predicted Label: {res['label']} | Conf: {res['confidence']:.3f} | Latency: {res['latency_ms']:.2f}ms | Fallback: {res['is_fallback']}")


--- LIVE INFERENCE DEMO ---
Query: kripaya rasoeeghar ki battiyaan band kar den
  -> Predicted Label: 39 | Conf: 0.884 | Latency: 19.99ms | Fallback: False
Query: कृपया रसोईघर की बत्तियां बंद कर दें
  -> Predicted Label: 39 | Conf: 0.940 | Latency: 18.93ms | Fallback: False
Query: kya kal subah ke liye koi alarm set hai mera bhai
  -> Predicted Label: 55 | Conf: 0.596 | Latency: 19.00ms | Fallback: True
Query: bina tel ke svadisht snaiks kaise pakaaen
  -> Predicted Label: 36 | Conf: 0.902 | Latency: 19.27ms | Fallback: False
Query: volyum star ko adhiktam mein badlen
  -> Predicted Label: 28 | Conf: 0.793 | Latency: 19.40ms | Fallback: False


## 8. Generate and Validate Final `submission.csv`
Generate test predictions with the exact calibrated probabilities and Tier-2 fallback.


In [8]:
adj_test_probs = test_probs * (1.0 / (prior ** tau))
final_preds = np.argmax(adj_test_probs, axis=1)

# Tier 2 open set rejection for anomalous long prose
is_long_prose = test_df['query_text'].apply(lambda x: len(x) > 250).values
final_preds[is_long_prose] = 55

submission = pd.DataFrame({
    'example_id': test_df['example_id'],
    'target': final_preds
})

# Formal Verification Assertions
assert len(submission) == 4912, f"Length mismatch: {len(submission)}"
assert (submission['example_id'].values == sample_sub['example_id'].values).all(), "ID mismatch!"
assert submission['target'].dtype in [np.int64, np.int32, int], "Target not integer!"
assert not submission.isnull().any().any(), "Null values detected!"
assert submission['target'].min() >= 0 and submission['target'].max() <= 55, "Out of bounds!"

submission.to_csv('submission.csv', index=False)
print("SUCCESS: submission.csv regenerated and strictly verified against all competition requirements.")
print(f"Class 55 fallback count in test: {(submission['target'] == 55).sum()}")
print("Top 5 predicted classes:\n", submission['target'].value_counts().head(5))


SUCCESS: submission.csv regenerated and strictly verified against all competition requirements.
Class 55 fallback count in test: 309
Top 5 predicted classes:
 target
48    320
55    309
47    287
12    281
44    276
Name: count, dtype: int64


## 9. Parameter Count & Full-Pipeline Latency Profiling
Measure the parameter footprint and end-to-end inference latency under 1,000 queries.


In [9]:
total_params = clf_prod.coef_.shape[0] * clf_prod.coef_.shape[1] + len(clf_prod.intercept_)
print(f"Total Model Parameters: {total_params:,} ({total_params/1e6:.2f}M) <= 300M -> COMPLIANT")

benchmark_queries = [
    "kripaya rasoeeghar ki battiyaan band kar den",
    "कृपया रसोईघर की बत्तियां बंद कर दें",
    "volyum star ko adhiktam mein badlen",
    "gravity kya hoti hai bata sakte ho please",
    "kya kal subah ke liye koi alarm set hai mera bhai",
    "mujhe hiphajurrahmaan ka mobail nanbar do"
]

latencies = []
for _ in range(100):
    for q in benchmark_queries:
        latencies.append(predict(q)['latency_ms'])

p50 = np.percentile(latencies, 50)
p95 = np.percentile(latencies, 95)
p99 = np.percentile(latencies, 99)

print(f"End-to-End Latency Profile (600 runs):")
print(f"  p50: {p50:.3f} ms")
print(f"  p95: {p95:.3f} ms")
print(f"  p99: {p99:.3f} ms")
print("Constraint Check: p95 < 50ms -> COMPLIANT")


Total Model Parameters: 4,433,912 (4.43M) <= 300M -> COMPLIANT


End-to-End Latency Profile (600 runs):
  p50: 20.616 ms
  p95: 22.742 ms
  p99: 23.768 ms
Constraint Check: p95 < 50ms -> COMPLIANT
